# Building Reliable RAG Systems — From First Principles

## Overview

Retrieval-Augmented Generation (RAG) connects a language model to an external knowledge base so it can answer questions grounded in real documents rather than relying on memorised (and potentially stale) parametric knowledge. In theory, RAG should eliminate hallucination — the model simply reads the relevant passage and paraphrases it. In practice, every stage of the pipeline is a potential failure point: the retriever might miss the right chunk, return a misleading one, or the generator might ignore the evidence entirely and confabulate an answer that *sounds* authoritative but is fabricated.

This notebook is a deep, hands-on exploration of **reliability engineering for RAG systems**. We will build a complete RAG pipeline from scratch — using only native Python, HuggingFace Transformers, FAISS, and sentence-transformers — and then systematically layer defensive mechanisms on top: confidence-based retrieval gates, hallucination detection, claim-context alignment, self-evaluation, and a full answer-validation pipeline that knows when to say "I don't know."

By the end you will have a battle-tested toolkit for shipping RAG systems that fail gracefully rather than silently.

## Technical Stack

| Component | Choice |
|---|---|
| **LLM** | `Qwen/Qwen3-8B` (4-bit NF4 quantisation) |
| **Embeddings** | `BAAI/bge-base-en-v1.5` via sentence-transformers |
| **Vector Store** | FAISS (faiss-cpu) |
| **Frameworks** | Native Python — no LangChain, no LlamaIndex, no OpenAI API |

## Setup

First, let's install dependencies, load the model, and define our `generate()` helper.

In [ ]:
# --- Cell 1: Install Dependencies ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy

In [ ]:
# --- Cell 2: Load LLM (Qwen3-8B, 4-bit NF4) ---
import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample, top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print("✅ LLM loaded:", MODEL_NAME)

In [ ]:
# --- Cell 3: Load Embedding Model (bge-base-en-v1.5) ---
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "BAAI/bge-base-en-v1.5"
embed_model = SentenceTransformer(EMBED_MODEL_NAME, cache_folder=CACHE_DIR)
print("✅ Embedding model loaded:", EMBED_MODEL_NAME)
print("   Embedding dimension:", embed_model.get_sentence_embedding_dimension())

In [ ]:
# --- Cell 4: generate() Quick Smoke Test ---
test_msg = [{"role": "user", "content": "In one sentence, what is retrieval-augmented generation?"}]
print(generate(test_msg, max_new_tokens=80))
print("\n✅ generate() helper is working.")

---
# Section 1: Why RAG Systems Fail

## A Taxonomy of RAG Failures

RAG systems can fail at **every stage** of their pipeline, and the failure modes are qualitatively different depending on where they occur. Understanding this taxonomy is the first step towards building reliable systems, because each failure mode demands a different defensive mechanism.

**1. Retrieval Failures** — The retriever returns chunks that are irrelevant, partially relevant, or missing the key information entirely. This happens when the query-document semantic gap is large (e.g., the user asks "What causes high blood pressure?" but the chunk uses the term "hypertension" without the layperson synonym), when the knowledge base simply doesn't contain the answer, or when the embedding model lacks domain-specific understanding. Retrieval failure is the single most common cause of bad RAG answers — garbage in, garbage out.

**2. Context Poisoning** — The retriever returns chunks that are *topically related* but contain misleading, outdated, or contradictory information. The model dutifully paraphrases the wrong evidence. This is especially dangerous because the answer looks well-grounded to a human reviewer — it has the right *style* even though the *content* is wrong.

**3. Generation Hallucination** — The model receives good evidence but ignores it, instead generating text from its parametric memory. This often happens when the model's prior belief is very strong (e.g., well-known facts) or when the context is long and the relevant passage is buried in the middle ("lost in the middle" effect).

**4. Format / Compliance Errors** — The answer is factually correct but doesn't comply with the requested format, length, or style constraints. In production systems with structured output requirements (JSON, citations, specific section headings), this is a significant reliability concern.

## The "Garbage In, Garbage Out" Principle

The most important insight in RAG reliability is this: **the generator can never be more reliable than the retriever**. A perfect language model given irrelevant context will either hallucinate (ignoring the context) or produce an irrelevant answer (following the context faithfully). There is no winning move.

This means that the *first* and *highest-leverage* investment in RAG reliability is always retrieval quality. Before you add re-rankers, chain-of-thought prompting, or self-consistency decoding, make sure your top-k chunks actually contain the answer. Everything else is damage control.

The implication for system design is that we need **quality gates** — automated checks at each pipeline stage that can detect when something has gone wrong and either retry, fall back, or refuse to answer. A system that says "I don't know" is infinitely more reliable than one that confidently fabricates an answer.

In [ ]:
# --- Visualise the RAG failure taxonomy ---
failure_taxonomy = {
    "Stage":        ["Retrieval", "Retrieval",   "Generation",       "Generation",     "Output"],
    "Failure Mode": ["Miss",     "Poison",       "Hallucination",    "Unfaithfulness", "Format Error"],
    "Description":  [
        "Relevant chunks not retrieved (semantic gap, missing data)",
        "Topically-related but misleading/contradictory chunks returned",
        "Model ignores context and generates from parametric memory",
        "Model partially uses context but adds unsupported claims",
        "Answer is correct but violates format/length/style constraints",
    ],
    "Severity":     ["Critical", "Critical",     "High",             "Medium",         "Low"],
    "Detectable?":  ["Yes — score thresholds", "Hard — needs verification",
                     "Yes — claim-context check", "Yes — claim-context check",
                     "Yes — regex/schema validation"],
}

# Pretty-print as a table
header = f"{'Stage':<13} {'Failure Mode':<17} {'Severity':<10} {'Detectable?':<30} Description"
print(header)
print("=" * 120)
for i in range(len(failure_taxonomy["Stage"])):
    print(f"{failure_taxonomy['Stage'][i]:<13} "
          f"{failure_taxonomy['Failure Mode'][i]:<17} "
          f"{failure_taxonomy['Severity'][i]:<10} "
          f"{failure_taxonomy['Detectable?'][i]:<30} "
          f"{failure_taxonomy['Description'][i]}")
print()
print("Key insight: The TWO critical failures are both in the retrieval stage.")
print("Invest in retrieval quality first — everything else is damage control.")

## Real-World Failure Examples

Let us build a small knowledge base and observe these failure modes in action. We will use a synthetic dataset about a fictional company's internal policies — small enough to fit in a notebook, rich enough to expose all four failure modes.

In [ ]:
# --- Build synthetic knowledge base ---
import numpy as np
import faiss

DOCUMENTS = [
    # HR Policies
    "Employees are entitled to 20 days of paid annual leave per calendar year. "
    "Unused leave can be carried over up to a maximum of 5 days into the next year. "
    "Leave requests must be submitted at least 2 weeks in advance through the HR portal.",

    "The company provides comprehensive health insurance covering medical, dental, and vision care. "
    "Coverage begins on the first day of employment. Dependents (spouse and children under 26) "
    "are automatically included at no additional cost.",

    "Remote work is permitted up to 3 days per week with manager approval. "
    "Employees must be available during core hours (10am-4pm) regardless of location. "
    "A home office stipend of $500 is provided annually for equipment purchases.",

    # Engineering Policies
    "All production deployments must go through a CI/CD pipeline with automated tests. "
    "Code coverage must exceed 80% for any pull request to be merged. "
    "Deployments to production are only permitted on Tuesday through Thursday.",

    "Security incidents must be reported to the security team within 1 hour of discovery. "
    "All customer data is encrypted at rest using AES-256 and in transit using TLS 1.3. "
    "Access to production databases requires two-factor authentication and manager approval.",

    "The engineering team follows a two-week sprint cycle with planning on Mondays "
    "and retrospectives on alternate Fridays. Story points use a Fibonacci scale. "
    "Each engineer is expected to complete 8-13 story points per sprint.",

    # Finance Policies
    "Expense reports must be submitted within 30 days of the expense date. "
    "Receipts are required for all expenses over $25. "
    "Travel expenses are reimbursed at the federal per diem rate for the destination city.",

    "The annual budget cycle runs from October to December. Department heads must submit "
    "budget proposals by November 1st. Capital expenditures over $10,000 require VP approval. "
    "Operating budgets are reviewed quarterly with variance analysis.",

    # Contradictory/Outdated chunk (deliberate for testing)
    "NOTE (SUPERSEDED - Jan 2024): Remote work was limited to 1 day per week. "
    "This policy has been updated — see current remote work policy for details. "
    "The old home office stipend was $200 per year.",
]

METADATA = [
    {"source": "HR Handbook", "section": "Leave Policy", "updated": "2024-06"},
    {"source": "HR Handbook", "section": "Health Insurance", "updated": "2024-01"},
    {"source": "HR Handbook", "section": "Remote Work", "updated": "2024-03"},
    {"source": "Engineering Wiki", "section": "Deployment", "updated": "2024-05"},
    {"source": "Engineering Wiki", "section": "Security", "updated": "2024-04"},
    {"source": "Engineering Wiki", "section": "Sprint Process", "updated": "2024-02"},
    {"source": "Finance Manual", "section": "Expenses", "updated": "2024-03"},
    {"source": "Finance Manual", "section": "Budget Cycle", "updated": "2024-01"},
    {"source": "HR Handbook", "section": "Remote Work (OUTDATED)", "updated": "2023-01"},
]

print(f"📚 Knowledge base: {len(DOCUMENTS)} documents")
for i, (doc, meta) in enumerate(zip(DOCUMENTS, METADATA)):
    print(f"  [{i}] {meta['source']} > {meta['section']} ({len(doc)} chars)")

In [ ]:
# --- Create FAISS index ---
# bge-base-en-v1.5 recommends prepending "Represent this sentence: " for retrieval
doc_embeddings = embed_model.encode(DOCUMENTS, normalize_embeddings=True, show_progress_bar=True)
dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)  # Inner product = cosine similarity (since normalised)
index.add(np.array(doc_embeddings, dtype=np.float32))
print(f"✅ FAISS index built: {index.ntotal} vectors, dimension={dimension}")

def retrieve(query, k=3):
    """Retrieve top-k documents with similarity scores."""
    query_embedding = embed_model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(np.array(query_embedding, dtype=np.float32), k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "text": DOCUMENTS[idx],
            "metadata": METADATA[idx],
            "score": float(score),
            "index": int(idx),
        })
    return results

# Quick test
results = retrieve("How many vacation days do employees get?")
for r in results:
    print(f"  Score: {r['score']:.4f} | {r['metadata']['section']}")
print("\n✅ Retrieval function working.")

## Reliability Metrics

Before we can improve reliability, we need to *measure* it. The RAG research community has converged on four core metrics:

| Metric | What It Measures | How We'll Estimate It |
|---|---|---|
| **Faithfulness** | Is the answer supported by the retrieved context? | LLM-as-judge: "Is every claim in the answer found in the context?" |
| **Relevance** | Does the answer actually address the question? | LLM-as-judge: "Does this answer the user's question?" |
| **Groundedness** | Are specific claims traceable to specific chunks? | Claim extraction + context matching |
| **Coverage** | Does the answer use all relevant information? | Compare answer claims vs. context claims |

In production systems these metrics would be computed by a separate evaluator model (often called a "judge"). In this notebook, we will use the same Qwen3-8B model in evaluator mode — this is a simplification, but the *pattern* transfers directly to production where you would use a dedicated judge model.

The key philosophical point: **reliability is not a binary property**. It exists on a spectrum, and our job is to push the system's operating point as far towards "reliable" as the application domain demands. A chatbot for restaurant recommendations can tolerate some imprecision; a medical QA system cannot.

In [ ]:
# --- Demonstrate failure modes with our knowledge base ---
print("=" * 70)
print("FAILURE MODE DEMONSTRATIONS")
print("=" * 70)

# 1. Retrieval miss — out-of-scope query
print("\n🔴 FAILURE MODE 1: Retrieval Miss (out-of-scope query)")
print("-" * 50)
results_miss = retrieve("What is the company's policy on pet-friendly offices?")
for r in results_miss:
    print(f"  Score: {r['score']:.4f} | {r['metadata']['section']}")
print("  → None of these chunks answer the question about pets!")

# 2. Context poisoning — contradictory information
print("\n🟠 FAILURE MODE 2: Context Poisoning (outdated chunk)")
print("-" * 50)
results_poison = retrieve("How many days per week can I work remotely?")
for r in results_poison:
    print(f"  Score: {r['score']:.4f} | {r['metadata']['section']} ({r['metadata']['updated']})")
print("  → The OUTDATED chunk competes with the current policy!")

# 3. Ambiguous query
print("\n🟡 FAILURE MODE 3: Ambiguous Query")
print("-" * 50)
results_ambig = retrieve("What are the approval requirements?")
for r in results_ambig:
    print(f"  Score: {r['score']:.4f} | {r['metadata']['section']}")
print("  → Multiple domains match — which 'approval' does the user mean?")

---
# Section 2: Retrieval Quality Gates

## Confidence Scoring: Using Similarity Scores as Quality Signals

The simplest and most powerful reliability mechanism in a RAG system is the **confidence gate** — using the retriever's similarity scores to decide whether the retrieved evidence is good enough to answer the question. If the best-matching chunk scores below a threshold, we should refuse to answer rather than risk hallucination.

This is directly analogous to a classifier's decision boundary. Just as you wouldn't deploy a classifier with 50% confidence on a medical diagnosis, you shouldn't let a RAG system generate an answer when the retrieval confidence is low. The key insight is that cosine similarity between query and document embeddings is a *noisy but informative* signal of whether the answer exists in the retrieved context.

Setting the threshold requires calibration. Too high, and the system refuses too many legitimate queries (low recall). Too low, and it passes through irrelevant context (low precision). In practice, we recommend starting with a threshold around 0.60-0.70 for bge-base-en-v1.5 (which produces normalised embeddings, so scores are cosine similarities in [0, 1]), then adjusting based on empirical evaluation on a held-out query set.

In [ ]:
# --- Implement retrieval with confidence filtering ---

def retrieve_with_confidence(query, k=3, min_score=0.65):
    """
    Retrieve top-k documents, but filter out any below min_score.
    Returns (results, is_confident) tuple.
    """
    query_embedding = embed_model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(np.array(query_embedding, dtype=np.float32), k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "text": DOCUMENTS[idx],
            "metadata": METADATA[idx],
            "score": float(score),
            "index": int(idx),
        })

    # Filter by minimum score
    confident_results = [r for r in results if r["score"] >= min_score]
    is_confident = len(confident_results) > 0

    return confident_results, is_confident, results  # Also return unfiltered for inspection

# Test on a well-scoped query
print("✅ WELL-SCOPED QUERY: 'How many vacation days do I get?'")
confident, is_conf, all_results = retrieve_with_confidence("How many vacation days do I get?", min_score=0.65)
for r in all_results:
    status = "✅ PASS" if r["score"] >= 0.65 else "❌ FILTERED"
    print(f"  {status} Score: {r['score']:.4f} | {r['metadata']['section']}")
print(f"  Confident: {is_conf} | Chunks passing gate: {len(confident)}/{len(all_results)}")

print()

# Test on an out-of-scope query
print("🚫 OUT-OF-SCOPE QUERY: 'What is the company policy on bringing pets to the office?'")
confident, is_conf, all_results = retrieve_with_confidence(
    "What is the company policy on bringing pets to the office?", min_score=0.65
)
for r in all_results:
    status = "✅ PASS" if r["score"] >= 0.65 else "❌ FILTERED"
    print(f"  {status} Score: {r['score']:.4f} | {r['metadata']['section']}")
print(f"  Confident: {is_conf} | Chunks passing gate: {len(confident)}/{len(all_results)}")
if not is_conf:
    print("  → System would REFUSE to answer. ✅ Correct behaviour!")

## Multi-Source Verification: The N-Source Agreement Check

A single high-scoring chunk might be a false positive — topically related but not actually answering the question. A much stronger signal is when **multiple independent chunks** agree on the answer. This is the RAG equivalent of the journalism principle: "If your mother says she loves you, get a second source."

The N-source agreement check requires that at least N of the top-K retrieved chunks are relevant to the query before the system proceeds to generation. This dramatically reduces the risk of context poisoning, because it's unlikely that multiple chunks will independently contain the *same* misleading information.

In practice, N=2 with K=3 is a good starting point: require at least 2 of the top 3 chunks to be relevant. For high-stakes applications (medical, legal, financial), you might require N=3 with K=5.

In [ ]:
# --- Implement N-source agreement check ---

def check_multi_source_agreement(results, min_sources=2, min_score=0.65):
    """
    Verify that at least min_sources chunks pass the confidence threshold.
    Returns (passing_chunks, agreement_met).
    """
    passing = [r for r in results if r["score"] >= min_score]
    agreement_met = len(passing) >= min_sources
    return passing, agreement_met

# Test: well-scoped query (should pass)
print("TEST 1: Well-scoped query")
_, _, all_r = retrieve_with_confidence("What health insurance does the company provide?", k=3, min_score=0.0)
passing, agreed = check_multi_source_agreement(all_r, min_sources=2, min_score=0.65)
for r in all_r:
    tag = "✅" if r["score"] >= 0.65 else "❌"
    print(f"  {tag} {r['score']:.4f} | {r['metadata']['section']}")
print(f"  Agreement: {agreed} ({len(passing)}/2 required)\n")

# Test: narrow query (may not pass)
print("TEST 2: Narrow query (specific to one document)")
_, _, all_r = retrieve_with_confidence("What is the code coverage requirement for pull requests?", k=3, min_score=0.0)
passing, agreed = check_multi_source_agreement(all_r, min_sources=2, min_score=0.65)
for r in all_r:
    tag = "✅" if r["score"] >= 0.65 else "❌"
    print(f"  {tag} {r['score']:.4f} | {r['metadata']['section']}")
print(f"  Agreement: {agreed} ({len(passing)}/2 required)")
if not agreed:
    print("  → Only 1 chunk is clearly relevant — consider single-source mode or lower threshold")

## Combining Confidence and Agreement into a Retrieval Gate

Now we combine both mechanisms into a single **retrieval quality gate** that makes a three-way decision:

1. **HIGH CONFIDENCE** — Multiple chunks agree and all score above threshold → proceed to generation
2. **MEDIUM CONFIDENCE** — One strong chunk but no multi-source agreement → proceed with a warning / reduced confidence label
3. **LOW CONFIDENCE** — No chunks pass the threshold → refuse to answer

This three-tier approach avoids the binary trap of "answer or refuse." The medium tier is especially useful because it allows the system to provide a hedged answer: "Based on limited information, …" rather than either hallucinating or refusing.

In [ ]:
# --- Complete Retrieval Quality Gate ---

def retrieval_quality_gate(query, k=3, min_score=0.65, min_sources=2):
    """
    Three-tier retrieval quality gate.
    Returns (results, confidence_level, message).
    """
    query_embedding = embed_model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(np.array(query_embedding, dtype=np.float32), k)

    all_results = []
    for score, idx in zip(scores[0], indices[0]):
        all_results.append({
            "text": DOCUMENTS[idx],
            "metadata": METADATA[idx],
            "score": float(score),
            "index": int(idx),
        })

    passing = [r for r in all_results if r["score"] >= min_score]

    if len(passing) >= min_sources:
        return passing, "HIGH", f"{len(passing)} chunks above threshold — multi-source agreement ✅"
    elif len(passing) == 1:
        return passing, "MEDIUM", "Single source only — proceed with caution ⚠️"
    else:
        return [], "LOW", "No chunks above threshold — refuse to answer 🚫"

# Demo on three query types
test_queries = [
    "How many days of annual leave do employees get?",
    "What is the sprint velocity expectation for engineers?",
    "What is the company's parental leave policy?",
]

for q in test_queries:
    results, level, msg = retrieval_quality_gate(q)
    print(f"Query: '{q}'")
    print(f"  Level: {level} | {msg}")
    for r in results:
        print(f"    Score: {r['score']:.4f} | {r['metadata']['section']}")
    print()

In [ ]:
# --- Threshold calibration: survey scores across all queries ---
calibration_queries = [
    "How many vacation days?",
    "Health insurance coverage",
    "Remote work policy",
    "Deployment rules",
    "Security incident reporting",
    "Sprint cycle details",
    "Expense report rules",
    "Budget approval process",
    "Maternity leave policy",       # Not in KB
    "Stock option vesting schedule", # Not in KB
    "Office pet policy",             # Not in KB
]

print("Threshold Calibration Survey")
print("=" * 80)
print(f"{'Query':<35} {'Top-1 Score':>11} {'Top-2 Score':>11} {'Top-3 Score':>11} {'In KB?':>7}")
print("-" * 80)

for q in calibration_queries:
    qe = embed_model.encode([q], normalize_embeddings=True)
    scores, indices = index.search(np.array(qe, dtype=np.float32), 3)
    in_kb = "Yes" if scores[0][0] > 0.70 else "No?"
    print(f"{q:<35} {scores[0][0]:>11.4f} {scores[0][1]:>11.4f} {scores[0][2]:>11.4f} {in_kb:>7}")

print()
print("Observation: queries answerable from our KB tend to have top-1 scores > 0.70.")
print("Queries NOT in the KB cluster below 0.60. A threshold around 0.65 separates them well.")

---
# Section 3: Hallucination Detection

## What Causes Hallucination in RAG?

Even when the retriever returns the perfect chunk, the generator can still produce unfaithful output. Hallucination in RAG has several root causes:

**1. Weak evidence + strong prior**: When the retrieved context is thin (e.g., a short or vague chunk), the model's parametric memory dominates. If the model "knows" a fact from pre-training that contradicts or extends the context, it may override the evidence. This is especially common for well-known topics where the model has strong priors.

**2. The "lost in the middle" effect**: Research by Liu et al. (2023) showed that language models pay disproportionate attention to the *beginning* and *end* of long contexts, often ignoring information in the middle. If the key evidence is in chunk #2 of 5, the model might miss it entirely and generate from its own knowledge instead.

**3. Extrapolation from partial evidence**: The model reads a chunk that says "employees get 20 days of leave" and confidently adds "plus 5 additional personal days" — information that isn't in the context but *seems* plausible. This is the most dangerous form of hallucination because it's mixed in with grounded claims.

**4. Instruction-following conflicts**: The system prompt says "answer helpfully" but the context doesn't contain enough information for a helpful answer. The model resolves this tension by fabricating content to fulfill the helpfulness objective.

## Citation Verification: Claim-Context Alignment

The gold standard for hallucination detection is **claim-level verification**: extract each factual claim from the generated answer, then check whether it can be traced back to a specific passage in the context. Claims that cannot be grounded are flagged as potential hallucinations.

This is computationally expensive (it requires multiple LLM calls), but it's the most reliable method available. In production, you might run this check asynchronously and flag answers for human review rather than blocking the response.

The process is:
1. **Extract claims** from the generated answer (each discrete factual assertion)
2. **For each claim**, check if it is **supported**, **contradicted**, or **not mentioned** in the context
3. **Compute a faithfulness score** = (supported claims) / (total claims)
4. If faithfulness < threshold, flag the answer for review or regenerate

In [ ]:
# --- Implement claim extraction ---

def extract_claims(answer):
    """Use the LLM to extract discrete factual claims from an answer."""
    messages = [
        {"role": "system", "content": (
            "You are a claim extractor. Given an answer, extract each discrete factual claim as a "
            "separate bullet point. Each claim should be a single, verifiable statement. "
            "Return ONLY the bullet list, nothing else. Use '- ' prefix for each claim."
        )},
        {"role": "user", "content": f"Extract claims from this answer:\n\n{answer}"},
    ]
    response = generate(messages, max_new_tokens=300, temperature=0.1, do_sample=True)
    claims = [line.strip().lstrip("- ").strip()
              for line in response.strip().split("\n")
              if line.strip().startswith("- ") or line.strip().startswith("* ")]
    return claims

# Demo
test_answer = (
    "Employees receive 20 days of paid annual leave per year. "
    "Up to 5 unused days can be carried over to the next year. "
    "Leave requests must be submitted 2 weeks in advance through the HR portal. "
    "Additionally, employees get 10 sick days per year."  # <-- This is hallucinated!
)
claims = extract_claims(test_answer)
print("Extracted claims:")
for i, c in enumerate(claims):
    print(f"  [{i}] {c}")
print(f"\nTotal claims extracted: {len(claims)}")

In [ ]:
# --- Implement claim-context alignment checker ---

def verify_claim(claim, context_chunks):
    """Check if a single claim is supported by the retrieved context."""
    context_text = "\n---\n".join([c["text"] for c in context_chunks])
    messages = [
        {"role": "system", "content": (
            "You are a fact-checking assistant. Given a CLAIM and a CONTEXT, determine if the claim "
            "is SUPPORTED, CONTRADICTED, or NOT MENTIONED in the context. "
            "Respond with exactly one word: SUPPORTED, CONTRADICTED, or NOT_MENTIONED."
        )},
        {"role": "user", "content": (
            f"CONTEXT:\n{context_text}\n\n"
            f"CLAIM: {claim}\n\n"
            "Verdict (SUPPORTED / CONTRADICTED / NOT_MENTIONED):"
        )},
    ]
    response = generate(messages, max_new_tokens=20, temperature=0.1, do_sample=True).strip().upper()
    # Parse — be tolerant of model output variations
    if "SUPPORT" in response:
        return "SUPPORTED"
    elif "CONTRADICT" in response:
        return "CONTRADICTED"
    else:
        return "NOT_MENTIONED"

# Retrieve context for the leave policy
context_chunks, _, _ = retrieve_with_confidence("annual leave policy", k=3, min_score=0.0)

print("Claim-Context Alignment Check")
print("=" * 70)
print(f"Context source: {context_chunks[0]['metadata']['section']}")
print()

for i, claim in enumerate(claims):
    verdict = verify_claim(claim, context_chunks)
    icon = {"SUPPORTED": "✅", "CONTRADICTED": "❌", "NOT_MENTIONED": "⚠️"}.get(verdict, "❓")
    print(f"  {icon} [{verdict:<14}] {claim}")

print()
print("Note: The claim about '10 sick days' should be flagged as NOT_MENTIONED —")
print("it was hallucinated and has no basis in our knowledge base.")

In [ ]:
# --- Compute faithfulness score ---

def compute_faithfulness(answer, context_chunks):
    """
    Full faithfulness pipeline: extract claims → verify each → compute score.
    Returns (score, claim_verdicts).
    """
    claims = extract_claims(answer)
    if not claims:
        return 1.0, []  # No claims = vacuously faithful

    verdicts = []
    for claim in claims:
        verdict = verify_claim(claim, context_chunks)
        verdicts.append({"claim": claim, "verdict": verdict})

    supported = sum(1 for v in verdicts if v["verdict"] == "SUPPORTED")
    score = supported / len(verdicts) if verdicts else 0.0
    return score, verdicts

# Test with the answer containing a hallucinated claim
context, _, _ = retrieve_with_confidence("annual leave policy", k=3, min_score=0.0)
score, verdicts = compute_faithfulness(test_answer, context)

print(f"Faithfulness Score: {score:.2%}")
print(f"  Supported:     {sum(1 for v in verdicts if v['verdict'] == 'SUPPORTED')}")
print(f"  Contradicted:  {sum(1 for v in verdicts if v['verdict'] == 'CONTRADICTED')}")
print(f"  Not mentioned: {sum(1 for v in verdicts if v['verdict'] == 'NOT_MENTIONED')}")

if score < 1.0:
    print("\n⚠️  Unfaithful claims detected — answer should be flagged for review.")

## Self-Evaluation: Confidence Calibration

A complementary approach to external verification is **self-evaluation**: asking the model to rate its *own* confidence in the answer given the context. This is faster than full claim verification (one LLM call instead of N) and surprisingly effective.

The idea is simple: after generating an answer, we ask the same model "On a scale of 1-5, how confident are you that this answer is fully supported by the provided context?" Models are imperfect at self-assessment, but research shows they are *directionally correct*: they tend to report lower confidence when the context is weak or the answer required extrapolation.

Self-evaluation is best used as a **fast filter** in combination with the more expensive claim verification. If self-evaluated confidence is high, skip full verification. If it's low, trigger the expensive check.

In [ ]:
# --- Implement self-evaluation pipeline ---

def self_evaluate_confidence(question, answer, context_chunks):
    """Ask the model to rate its confidence that the answer is grounded in context."""
    context_text = "\n---\n".join([c["text"] for c in context_chunks])
    messages = [
        {"role": "system", "content": (
            "You are evaluating whether an AI-generated answer is fully supported by the provided context. "
            "Rate your confidence on a scale of 1-5:\n"
            "  1 = Answer has NO support in context (completely made up)\n"
            "  2 = Answer has WEAK support (some relevant info but key claims unsupported)\n"
            "  3 = Answer is PARTIALLY supported (main point correct but some details unverified)\n"
            "  4 = Answer is MOSTLY supported (nearly all claims grounded in context)\n"
            "  5 = Answer is FULLY supported (every claim traceable to context)\n\n"
            "Respond with ONLY the number (1-5) followed by a brief explanation."
        )},
        {"role": "user", "content": (
            f"CONTEXT:\n{context_text}\n\n"
            f"QUESTION: {question}\n\n"
            f"ANSWER: {answer}\n\n"
            "Confidence rating:"
        )},
    ]
    response = generate(messages, max_new_tokens=100, temperature=0.1, do_sample=True)
    # Extract numeric rating
    for char in response:
        if char.isdigit() and 1 <= int(char) <= 5:
            return int(char), response.strip()
    return 3, response.strip()  # Default to 3 if parsing fails

# Test with grounded answer
grounded_answer = (
    "Employees receive 20 days of paid annual leave per year. "
    "Up to 5 unused days can be carried over. "
    "Leave requests must be submitted 2 weeks in advance."
)
context, _, _ = retrieve_with_confidence("annual leave policy", k=3, min_score=0.0)

print("TEST 1: Fully grounded answer")
rating, explanation = self_evaluate_confidence("How much vacation do I get?", grounded_answer, context)
print(f"  Rating: {rating}/5")
print(f"  Explanation: {explanation[:150]}...")

print()

# Test with partially hallucinated answer
print("TEST 2: Answer with hallucinated claim (10 sick days)")
rating2, explanation2 = self_evaluate_confidence("How much vacation do I get?", test_answer, context)
print(f"  Rating: {rating2}/5")
print(f"  Explanation: {explanation2[:150]}...")

print()
print("Expected: grounded answer should score 4-5, hallucinated answer should score 2-3.")

---
# Section 4: Answer Validation Pipeline

## The Verify-Then-Answer Pattern

Now we bring everything together into a single **answer validation pipeline** that implements the "verify-then-answer" pattern. Instead of the naive `retrieve → generate → return` pipeline, our reliable pipeline follows this flow:

```
1. RETRIEVE  → Fetch top-K chunks from FAISS
2. GATE      → Apply retrieval quality gate (confidence + multi-source)
3. GENERATE  → If gate passes, generate answer with context
4. VERIFY    → Self-evaluate confidence; if low, run claim verification
5. DECIDE    → Return answer (with confidence label) OR refuse gracefully
```

This is a **defense-in-depth** strategy. Each layer catches failures that the previous layer missed. The retrieval gate catches missing/irrelevant context. Self-evaluation catches obvious hallucinations cheaply. Claim verification catches subtle unfaithfulness. And the final decision layer ensures we always communicate our confidence level to the user.

The key design principle is **graceful degradation**: rather than a binary "answer or crash," the system produces a spectrum of responses ranging from "here's a confident answer with sources" to "I found some related information but I'm not sure it answers your question" to "I don't have enough information to answer this."

In [ ]:
# --- Core RAG generation function ---

def rag_generate(question, context_chunks, hedged=False):
    """Generate a RAG answer from context chunks."""
    context_text = "\n---\n".join([
        f"[Source: {c['metadata']['source']} > {c['metadata']['section']}]\n{c['text']}"
        for c in context_chunks
    ])

    hedge_prefix = (
        "Note: I have limited evidence for this answer. "
        if hedged else ""
    )

    messages = [
        {"role": "system", "content": (
            "You are a helpful assistant that answers questions based ONLY on the provided context. "
            "Do not use any knowledge beyond what is explicitly stated in the context. "
            "If the context does not contain enough information to fully answer the question, "
            "say so explicitly rather than guessing."
        )},
        {"role": "user", "content": (
            f"CONTEXT:\n{context_text}\n\n"
            f"QUESTION: {question}\n\n"
            f"{hedge_prefix}Answer:"
        )},
    ]
    return generate(messages, max_new_tokens=256, temperature=0.3, do_sample=True)

# Quick test
context, _, _ = retrieve_with_confidence("How many vacation days?", k=3, min_score=0.0)
answer = rag_generate("How many vacation days do employees get?", context)
print("Question: How many vacation days do employees get?")
print(f"Answer: {answer}")

In [ ]:
# --- Complete Answer Validation Pipeline ---

def reliable_rag(question, k=3, min_score=0.65, min_sources=2,
                 self_eval_threshold=3, faithfulness_threshold=0.75,
                 verbose=True):
    """
    Full reliable RAG pipeline with layered validation.
    Returns dict with answer, confidence, metadata.
    """
    result = {
        "question": question,
        "answer": None,
        "confidence": None,
        "retrieval_level": None,
        "faithfulness_score": None,
        "self_eval_rating": None,
        "refused": False,
        "chunks_used": 0,
    }

    # --- Step 1: Retrieve ---
    chunks, level, gate_msg = retrieval_quality_gate(question, k=k, min_score=min_score, min_sources=min_sources)
    result["retrieval_level"] = level
    result["chunks_used"] = len(chunks)
    if verbose:
        print(f"[RETRIEVE] {gate_msg}")

    # --- Step 2: Gate check ---
    if level == "LOW":
        result["answer"] = (
            "I don't have enough information in my knowledge base to answer this question. "
            "Please try rephrasing or ask about a topic covered in our company policies."
        )
        result["confidence"] = "REFUSED"
        result["refused"] = True
        if verbose:
            print(f"[GATE] REFUSED — no relevant context found.")
        return result

    # --- Step 3: Generate ---
    hedged = (level == "MEDIUM")
    answer = rag_generate(question, chunks, hedged=hedged)
    result["answer"] = answer
    if verbose:
        print(f"[GENERATE] Answer produced ({len(answer)} chars), hedged={hedged}")

    # --- Step 4: Self-evaluate ---
    self_rating, self_explanation = self_evaluate_confidence(question, answer, chunks)
    result["self_eval_rating"] = self_rating
    if verbose:
        print(f"[SELF-EVAL] Rating: {self_rating}/5")

    # --- Step 5: Full verification if self-eval is low ---
    if self_rating <= self_eval_threshold:
        if verbose:
            print(f"[VERIFY] Self-eval below threshold ({self_eval_threshold}) — running claim verification...")
        faith_score, verdicts = compute_faithfulness(answer, chunks)
        result["faithfulness_score"] = faith_score
        if verbose:
            print(f"[VERIFY] Faithfulness: {faith_score:.2%}")

        if faith_score < faithfulness_threshold:
            result["confidence"] = "LOW"
            result["answer"] = (
                f"⚠️ Low confidence answer (faithfulness: {faith_score:.0%}):\n\n{answer}\n\n"
                "Note: Some claims in this answer could not be verified against the source documents. "
                "Please verify independently."
            )
            if verbose:
                print(f"[DECIDE] LOW confidence — answer flagged with warning.")
            return result

    # --- Step 6: Assign final confidence ---
    if level == "HIGH" and self_rating >= 4:
        result["confidence"] = "HIGH"
    elif level == "MEDIUM" or self_rating == 3:
        result["confidence"] = "MEDIUM"
    else:
        result["confidence"] = "HIGH"

    if verbose:
        print(f"[DECIDE] Final confidence: {result['confidence']}")

    return result

# Demo
print("=" * 70)
print("RELIABLE RAG PIPELINE — DEMO")
print("=" * 70)
result = reliable_rag("How many days of annual leave do employees get?")
print(f"\n📋 Answer: {result['answer']}")
print(f"📊 Confidence: {result['confidence']}")

## Graceful Degradation: When to Refuse

The refusal mechanism is one of the most important features of a reliable RAG system. In many production deployments, a confident wrong answer is *far worse* than no answer at all. Consider a medical QA system that confidently tells a patient the wrong dosage, or a legal assistant that fabricates a precedent.

Our pipeline has three refusal triggers:
1. **Retrieval gate refusal** — no relevant chunks found (fast, cheap)
2. **Faithfulness refusal** — answer generated but claims couldn't be verified (slow, thorough)
3. **Hedged response** — not a full refusal, but a clearly marked low-confidence answer

The refusal message should be *helpful*, not just a dead end. It should suggest what the user can do: rephrase the query, ask about a related topic, or escalate to a human expert.

In [ ]:
# --- Demonstrate refusal behaviour ---
print("=" * 70)
print("REFUSAL DEMONSTRATIONS")
print("=" * 70)

refusal_queries = [
    "What is the company's parental leave policy?",
    "How do I request a sabbatical?",
    "What are the stock option vesting terms?",
]

for q in refusal_queries:
    print(f"\nQuery: '{q}'")
    result = reliable_rag(q, verbose=False)
    print(f"  Confidence: {result['confidence']}")
    print(f"  Refused: {result['refused']}")
    print(f"  Answer: {result['answer'][:150]}...")
    print()

In [ ]:
# --- Demonstrate the full spectrum: high, medium, low confidence + refusal ---
spectrum_queries = [
    ("How many vacation days do employees get?", "Should be HIGH confidence"),
    ("What is the sprint velocity expectation?", "Should be MEDIUM or HIGH"),
    ("Can I work remotely and what was the old policy?", "May retrieve contradictory info"),
    ("What is the paternity leave policy?", "Should be REFUSED — not in KB"),
]

print("=" * 70)
print("CONFIDENCE SPECTRUM DEMONSTRATION")
print("=" * 70)

for query, expectation in spectrum_queries:
    print(f"\n{'─' * 60}")
    print(f"Query: {query}")
    print(f"Expected: {expectation}")
    print(f"{'─' * 60}")
    result = reliable_rag(query, verbose=True)
    print(f"\n→ Final confidence: {result['confidence']}")
    print(f"→ Answer preview: {result['answer'][:200]}...")

---
# Section 5: Robustness Testing

## Building a Test Suite for RAG Reliability

A reliable RAG system needs systematic testing — not just a few cherry-picked examples. We need to stress-test with a diverse set of queries that probe different failure modes: exact-match queries, paraphrased queries, ambiguous queries, multi-hop questions, adversarial queries, and out-of-scope queries.

Each test case should have an **expected behaviour** (answer correctly, refuse, hedge) and a **ground truth** (the expected answer or the absence of one). This lets us compute reliability metrics across the suite and identify systematic weaknesses.

The test suite below covers six categories:
1. **Direct** — straightforward factual questions with clear answers in the KB
2. **Paraphrased** — same questions rephrased differently (tests embedding robustness)
3. **Ambiguous** — questions that could refer to multiple topics
4. **Multi-hop** — questions requiring synthesis across multiple chunks
5. **Adversarial** — questions designed to trigger hallucination
6. **Out-of-scope** — questions with no answer in the KB

In [ ]:
# --- Define test suite ---
TEST_SUITE = [
    # Direct questions
    {"query": "How many days of paid leave do employees get per year?",
     "category": "direct", "expected": "answer", "ground_truth": "20 days"},
    {"query": "What encryption standard is used for customer data at rest?",
     "category": "direct", "expected": "answer", "ground_truth": "AES-256"},
    {"query": "What is the deadline for submitting budget proposals?",
     "category": "direct", "expected": "answer", "ground_truth": "November 1st"},

    # Paraphrased questions
    {"query": "How much annual vacation time is given to staff members?",
     "category": "paraphrased", "expected": "answer", "ground_truth": "20 days"},
    {"query": "On which days of the week are production releases allowed?",
     "category": "paraphrased", "expected": "answer", "ground_truth": "Tuesday through Thursday"},
    {"query": "What's the maximum receipt-free expense amount?",
     "category": "paraphrased", "expected": "answer", "ground_truth": "$25"},

    # Ambiguous questions
    {"query": "What are the approval requirements?",
     "category": "ambiguous", "expected": "hedge",
     "ground_truth": "Multiple possible — DB access, capital expenses, remote work"},
    {"query": "What is the deadline?",
     "category": "ambiguous", "expected": "hedge",
     "ground_truth": "Multiple possible — expense reports, budget proposals, leave requests"},

    # Multi-hop questions
    {"query": "If I want to deploy code on Friday and work remotely, is that allowed?",
     "category": "multi-hop", "expected": "answer",
     "ground_truth": "Deployments only Tue-Thu; remote up to 3 days/week with approval"},

    # Adversarial — designed to trigger hallucination
    {"query": "How many sick days do employees get?",
     "category": "adversarial", "expected": "refuse",
     "ground_truth": "NOT IN KB — should refuse or hedge"},
    {"query": "What is the company's policy on crypto investments?",
     "category": "adversarial", "expected": "refuse",
     "ground_truth": "NOT IN KB — should refuse"},

    # Out-of-scope
    {"query": "What is the CEO's salary?",
     "category": "out_of_scope", "expected": "refuse", "ground_truth": "NOT IN KB"},
    {"query": "Explain quantum computing.",
     "category": "out_of_scope", "expected": "refuse", "ground_truth": "NOT IN KB"},
    {"query": "What is the office pet policy?",
     "category": "out_of_scope", "expected": "refuse", "ground_truth": "NOT IN KB"},
]

print(f"Test suite: {len(TEST_SUITE)} test cases")
from collections import Counter
cats = Counter(t["category"] for t in TEST_SUITE)
for cat, count in sorted(cats.items()):
    print(f"  {cat}: {count} cases")

In [ ]:
# --- Run the test suite ---
print("=" * 80)
print("ROBUSTNESS TEST SUITE EXECUTION")
print("=" * 80)

test_results = []

for i, test in enumerate(TEST_SUITE):
    print(f"\n[{i+1}/{len(TEST_SUITE)}] ({test['category']}) {test['query']}")

    result = reliable_rag(test["query"], verbose=False)

    # Determine actual behaviour
    if result["refused"]:
        actual_behaviour = "refuse"
    elif result["confidence"] == "LOW":
        actual_behaviour = "hedge"
    elif result["confidence"] == "MEDIUM":
        actual_behaviour = "hedge"
    else:
        actual_behaviour = "answer"

    # Check if behaviour matches expectation
    correct = actual_behaviour == test["expected"]
    # Be lenient: "hedge" when expecting "answer" is acceptable (conservative)
    if test["expected"] == "answer" and actual_behaviour == "hedge":
        correct = True  # Conservative is OK
    # "refuse" when expecting "hedge" is also acceptable
    if test["expected"] == "hedge" and actual_behaviour == "refuse":
        correct = True

    icon = "✅" if correct else "❌"
    print(f"  {icon} Expected: {test['expected']}, Got: {actual_behaviour} "
          f"(confidence={result['confidence']})")
    print(f"     Ground truth: {test['ground_truth']}")
    print(f"     Answer: {str(result['answer'])[:120]}...")

    test_results.append({
        **test,
        "actual_behaviour": actual_behaviour,
        "confidence": result["confidence"],
        "correct": correct,
        "faithfulness": result.get("faithfulness_score"),
    })

In [ ]:
# --- Compute reliability metrics ---
print("\n" + "=" * 70)
print("RELIABILITY METRICS")
print("=" * 70)

total = len(test_results)
correct = sum(1 for t in test_results if t["correct"])
print(f"\nOverall Accuracy: {correct}/{total} ({correct/total:.0%})")

# Per-category breakdown
print(f"\n{'Category':<15} {'Correct':>8} {'Total':>6} {'Accuracy':>9}")
print("-" * 42)
categories = sorted(set(t["category"] for t in test_results))
for cat in categories:
    cat_results = [t for t in test_results if t["category"] == cat]
    cat_correct = sum(1 for t in cat_results if t["correct"])
    print(f"{cat:<15} {cat_correct:>8} {len(cat_results):>6} {cat_correct/len(cat_results):>9.0%}")

# Refusal accuracy
should_refuse = [t for t in test_results if t["expected"] in ("refuse",)]
actually_refused = [t for t in should_refuse if t["actual_behaviour"] in ("refuse", "hedge")]
print(f"\nRefusal Accuracy: {len(actually_refused)}/{len(should_refuse)} "
      f"({len(actually_refused)/len(should_refuse):.0%}) — correctly refused or hedged on out-of-scope queries")

# False refusal rate
should_answer = [t for t in test_results if t["expected"] == "answer"]
false_refusals = [t for t in should_answer if t["actual_behaviour"] == "refuse"]
print(f"False Refusal Rate: {len(false_refusals)}/{len(should_answer)} "
      f"({len(false_refusals)/len(should_answer):.0%}) — incorrectly refused answerable queries")

print("\n💡 Key insight: a system that sometimes over-refuses is SAFER than one that sometimes hallucinates.")
print("   The 'cost of silence' is almost always less than the 'cost of confabulation.'")

## Edge Cases: Empty Retrieval, Contradictory Sources, Paraphrased Questions

Three edge cases deserve special attention in production RAG systems:

**Empty retrieval** occurs when the FAISS index returns no results above the threshold. Our pipeline handles this gracefully with the retrieval gate, but it's worth noting that this can happen even for in-scope queries if the embedding model's representation of the query is far from any document. Solutions include query expansion, hypothetical document embeddings (HyDE), and spelling correction.

**Contradictory sources** are the hardest edge case. When two chunks provide conflicting information (as in our outdated remote-work policy example), the model must either choose one or acknowledge the contradiction. Our multi-source agreement check helps, but a more robust solution would include timestamp-based conflict resolution or explicit contradiction detection.

**Paraphrased questions** test the embedding model's semantic understanding. A good embedding model should map "How many vacation days?" and "What is the annual leave allowance?" to similar vectors. Our test suite includes paraphrased variants to verify this.

---
# Section 6: Synthesis — The Production Reliability Playbook

## The Reliability Checklist

Every production RAG system should implement these defences, in order of priority:

| Priority | Defence | Cost | Impact |
|---|---|---|---|
| **P0** | Retrieval confidence threshold | Negligible | Prevents ~60% of bad answers |
| **P0** | Graceful refusal ("I don't know") | Negligible | Eliminates worst-case failures |
| **P1** | Multi-source agreement | Low | Catches single-source false positives |
| **P1** | Self-evaluation | 1 extra LLM call | Fast hallucination screening |
| **P2** | Claim-context verification | N extra LLM calls | Gold-standard faithfulness |
| **P2** | Timestamp-based conflict resolution | Low | Handles outdated documents |
| **P3** | Query expansion / rewriting | 1 extra LLM call | Improves recall for tricky queries |
| **P3** | Human-in-the-loop for low-confidence | High | Ultimate safety net |

The key insight is that the first two defences (P0) are essentially free and prevent the majority of failures. You should implement them before doing anything else.

## Defence in Depth: Layered Validation

The architecture we've built implements a **defence in depth** strategy, inspired by security engineering. Each layer catches failures that slip through the previous one:

```
Layer 0: Embedding quality    ← Choose a good embedding model for your domain
Layer 1: Retrieval gate       ← Score threshold + multi-source agreement
Layer 2: Prompt engineering    ← "Answer ONLY from context" instruction
Layer 3: Self-evaluation      ← Fast confidence check (1 LLM call)
Layer 4: Claim verification   ← Full faithfulness audit (N LLM calls)
Layer 5: User-facing labels   ← Communicate confidence to the user
Layer 6: Human escalation     ← Route low-confidence answers to experts
```

No single layer is perfect. The embedding model will sometimes miss relevant chunks. The retrieval gate will sometimes pass irrelevant ones. The prompt instruction will sometimes be ignored. Self-evaluation will sometimes be overconfident. But the *combination* of all layers produces a system that is far more reliable than any single mechanism.

## Monitoring and Alerting

In production, reliability is not a one-time achievement — it requires continuous monitoring. Track these metrics over time:

- **Refusal rate**: If it spikes, your knowledge base may have gaps for new query patterns
- **Average retrieval score**: If it drifts downward, your embeddings may need retraining
- **Self-evaluation distribution**: If confidence drops, your prompts or data may need updating
- **Faithfulness scores**: The ultimate measure — sample and verify weekly

Set alerts for anomalies. A sudden increase in refusals might indicate a knowledge base update broke something. A sudden decrease in refusals might indicate the threshold was accidentally lowered.

## When Reliability Matters Most

Not all RAG applications require the same level of reliability. The cost of implementing and running verification layers is real (latency, compute, complexity), so match your investment to your risk profile:

| Domain | Risk Level | Minimum Defences |
|---|---|---|
| **Healthcare / Clinical** | Critical | All layers + human review + audit trail |
| **Legal / Compliance** | Critical | All layers + citation verification + human review |
| **Finance / Trading** | High | All layers + timestamp resolution + real-time monitoring |
| **Customer Support** | Medium | P0 + P1 + self-evaluation |
| **Internal Knowledge Base** | Medium | P0 + P1 |
| **Entertainment / Chat** | Low | P0 (confidence threshold + refusal) |

The principle is simple: **the cost of a wrong answer determines the investment in reliability**. When a wrong answer can harm someone's health, freedom, or finances, every layer of defence is justified. When the worst case is a mildly unhelpful chatbot response, a simpler pipeline suffices.

In [ ]:
# --- Final summary: print the complete defence stack we built ---
print("=" * 70)
print("🛡️  RELIABLE RAG — COMPLETE DEFENCE STACK SUMMARY")
print("=" * 70)

defences = [
    ("FAISS + bge-base-en-v1.5",       "Semantic retrieval with normalised cosine similarity"),
    ("Confidence threshold (0.65)",      "Filters out low-relevance chunks before generation"),
    ("Multi-source agreement (N=2)",     "Requires ≥2 supporting chunks for full confidence"),
    ("Three-tier retrieval gate",        "HIGH / MEDIUM / LOW confidence routing"),
    ("Context-only system prompt",       "Instructs model to use ONLY provided evidence"),
    ("Self-evaluation (1-5 rating)",     "Fast confidence screening — 1 LLM call"),
    ("Claim extraction + verification",  "Gold-standard faithfulness — N LLM calls"),
    ("Graceful refusal",                 "Says 'I don't know' when evidence is insufficient"),
    ("Hedged responses",                 "Marks low-confidence answers with warnings"),
    ("Confidence labels",                "Communicates reliability level to the user"),
]

for i, (name, desc) in enumerate(defences):
    print(f"  Layer {i}: {name}")
    print(f"           {desc}")
    print()

print("Each layer catches failures that slip through the previous one.")
print("Together, they form a defence-in-depth architecture for reliable RAG.")
print()
print("🎓 Key takeaway: reliability is not a feature — it's an architecture.")
print("   Build it in from the start, and your RAG system will earn user trust.")